# Week 2 – Data Collection, Cleaning and Preprocessing

The dataset used here is a **simulated logistics dataset** for educational and analytical purposes.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/simulated_logistics_data.csv")
print("Shape:", df.shape)
df.head()

## 1. Inspect the Dataset

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 2. Data Quality Checks

The preprocessing stage checks for missing values, duplicates, incorrect data types,
inconsistent categories, outliers, and impossible values.

In [ ]:
# Standardize categorical values
df["Vehicle_Type"] = df["Vehicle_Type"].astype(str).str.strip().str.title()
df["Traffic_Level"] = df["Traffic_Level"].astype(str).str.strip().str.title()
df["Weather"] = df["Weather"].astype(str).str.strip().str.title()
df["Delivery_Priority"] = df["Delivery_Priority"].astype(str).str.strip().str.title()

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Convert numeric columns explicitly
numeric_cols = [
    "Distance_km", "Package_Weight_kg", "Stops_Count",
    "Warehouse_Delay_Min", "Driver_Experience_Years",
    "Delivery_Time_Hours"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Cleaned shape:", df.shape)

## 3. Missing Values

In [ ]:
# Median imputation for numeric columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print(df.isna().sum())

## 4. Outlier Detection using IQR

In [ ]:
Q1 = df["Delivery_Time_Hours"].quantile(0.25)
Q3 = df["Delivery_Time_Hours"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower limit:", lower)
print("Upper limit:", upper)

# Flag rather than automatically delete observations
df["Delivery_Time_Outlier"] = (
    (df["Delivery_Time_Hours"] < lower) |
    (df["Delivery_Time_Hours"] > upper)
)

print("Potential outliers:", df["Delivery_Time_Outlier"].sum())

## 5. Save the Processed Dataset

In [ ]:
processed = df.drop(columns=["Delivery_Time_Outlier"])

processed.to_csv(
    "../data/processed/cleaned_logistics_data.csv",
    index=False
)

print("Processed dataset saved.")